# 🌍 Project 25 — Resample Image Resolution

## Goal

Convert an image to a different spatial resolution using different resampling methods.

You'll learn the three most common methods:

- **Nearest Neighbor**
- **Bilinear**
- **Bicubic**

---

# Why is this useful in Remote Sensing?

Resampling is commonly used for:

- 🛰 Comparing different satellite datasets
- 🤖 Preparing data for machine learning
- 🗺 Combining multiple sensors
- 🌍 Creating maps with a consistent resolution
- 📊 Image preprocessing

Almost every remote sensing workflow involves resampling at some stage.

---

# New Skill: Resampling Pixels

Previously:

```text
Original Image
      ↓
Display
```

Today:

```text
Original Image
      ↓
Choose Resampling Method
      ↓
Create New Pixel Values
      ↓
Display
```

---

# Connect to Previous Lessons

| Project | Skill |
|---------|-------|
| 22 | Median Composite |
| 23 | Mosaic |
| 24 | Compare Satellite Sensors |
| **25** | Resampling Images |

Project 24 taught you that different sensors have different resolutions.

Project 25 teaches you how to work with those differences.

---

# What is Resampling?

Imagine a satellite image made of pixels.

Original image:

```text
🟩 🟩 🟩
🟩 🟩 🟩
🟩 🟩 🟩
```

Suppose we want smaller pixels.

Earth Engine must estimate the new values.

Different methods estimate them differently.

---

# The Three Resampling Methods

## 1. Nearest Neighbor

```text
Old Pixel

🟩

↓

New Pixels

🟩 🟩
🟩 🟩
```

The new pixels simply copy the value of the nearest original pixel.

### Advantages

- Fast
- Original values are preserved

### Best for

- Land cover
- Classification maps
- Categorical data

---

## 2. Bilinear

```text
Pixel A ----- Pixel B
      \     /
       New
      /     \
Pixel C ----- Pixel D
```

The new pixel is calculated from the **four nearest pixels**.

### Advantages

- Smoother image
- Good balance between speed and quality

### Best for

- Satellite imagery
- NDVI
- NDWI
- DEMs

---

## 3. Bicubic

```text
Many Neighboring Pixels
          ↓
 Complex Interpolation
          ↓
 Very Smooth Image
```

Bicubic uses **16 neighboring pixels**.

### Advantages

- Very smooth
- Better visual appearance

### Disadvantages

- Slower
- Slightly changes original values

### Best for

- Visualization
- Maps for presentations

---

# Workflow

```text
Satellite Image
       ↓
Choose Resampling Method
       ↓
Nearest
or
Bilinear
or
Bicubic
       ↓
Display
```

---

# Step 1 — Define the Study Area

We'll continue using Berlin.

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.35],
    [13.65,52.35],
    [13.65,52.65],
    [13.20,52.65],
    [13.20,52.35]
  ]
]);
```

---

# Step 2 — Load a Sentinel-2 Image

```javascript
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate('2024-06-01','2024-06-30')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(studyArea);
```

---

# Step 3 — Apply Different Resampling Methods

### Nearest Neighbor

```javascript
var nearest = image.resample('nearest');
```

---

### Bilinear

```javascript
var bilinear = image.resample('bilinear');
```

---

### Bicubic

```javascript
var bicubic = image.resample('bicubic');
```

Notice something.

The image size doesn't immediately change.

`resample()` tells Earth Engine **how to calculate pixel values whenever the image is displayed or reprojected at another resolution.**

---

# Step 4 — Visualization

```javascript
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};
```

---

# Step 5 — Display

```javascript
Map.centerObject(studyArea,10);

Map.addLayer(
  nearest,
  trueColor,
  'Nearest Neighbor'
);

Map.addLayer(
  bilinear,
  trueColor,
  'Bilinear'
);

Map.addLayer(
  bicubic,
  trueColor,
  'Bicubic'
);
```

Zoom in closely.

The visual differences become more noticeable at high zoom levels.

---

# ⭐ Bonus — Compare with the Original Image

```javascript
Map.addLayer(
  image,
  trueColor,
  'Original'
);
```

Turn the layers on and off.

Ask yourself:

- Which image looks smoother?
- Which one keeps the sharpest edges?
- Which would you use for a land cover map?

---

# Complete Code

```javascript
// 1. Study Area
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.35],
    [13.65,52.35],
    [13.65,52.65],
    [13.20,52.65],
    [13.20,52.35]
  ]
]);

// 2. Load Image
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate('2024-06-01','2024-06-30')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(studyArea);

// 3. Resampling
var nearest = image.resample('nearest');

var bilinear = image.resample('bilinear');

var bicubic = image.resample('bicubic');

// 4. Visualization
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};

// 5. Display
Map.centerObject(studyArea,10);

Map.addLayer(
  image,
  trueColor,
  'Original'
);

Map.addLayer(
  nearest,
  trueColor,
  'Nearest Neighbor'
);

Map.addLayer(
  bilinear,
  trueColor,
  'Bilinear'
);

Map.addLayer(
  bicubic,
  trueColor,
  'Bicubic'
);
```

---

# Things to Remember

## 1. `resample()`

`resample()` specifies **how Earth Engine calculates new pixel values** when an image is displayed or transformed to a different resolution.

It does **not** immediately change the pixel size.

---

## 2. Choosing the Right Method

| Method | Best For |
|---------|----------|
| **Nearest Neighbor** | Land cover, classification maps, categorical data |
| **Bilinear** | Satellite imagery, NDVI, NDWI, DEMs |
| **Bicubic** | Visualization and presentation-quality maps |

A good rule of thumb:

- **Categorical data → Nearest Neighbor**
- **Continuous data → Bilinear**
- **Best visual appearance → Bicubic**

---

## 3. Continuous vs Categorical Data

This is one of the most important concepts in remote sensing.

### Continuous Data

Values can take many numbers.

Examples:

- NDVI
- NDWI
- Elevation
- Surface temperature
- Reflectance

These should usually use **bilinear** or **bicubic** interpolation.

---

### Categorical Data

Each value represents a class.

Examples:

- Forest
- Water
- Urban
- Cropland

These should use **nearest neighbor** so class labels are not changed.

---

# Common Beginner Mistakes

### ❌ Mistake 1: Using bilinear for land cover maps.

Land cover classes are labels (10, 20, 30, etc.), not measurements. Interpolating them can create meaningless values.

---

### ❌ Mistake 2: Expecting `resample()` to change the image resolution immediately.

`resample()` only defines the interpolation method. The actual resampling occurs when Earth Engine needs to display or reproject the image.

---

### ❌ Mistake 3: Thinking bicubic is always the best.

Bicubic creates smoother images, but it also modifies pixel values more than bilinear. For scientific analysis, bilinear is often the safer choice for continuous data.

---

# Your Practice (Project 25.1)

Compare resampling methods using a Sentinel-2 image.

## Requirements

1. Create a study area.
2. Load a Sentinel-2 image.
3. Create three versions using:
   - `nearest`
   - `bilinear`
   - `bicubic`
4. Display all three images.
5. Zoom in and compare their appearance.

Before writing the code, answer these two questions.

### Question 1

Why is **Nearest Neighbor** recommended for a land cover map, while **Bilinear** is usually preferred for NDVI?

---

### Question 2

Does the following line of code immediately change the image resolution?

```javascript
var bilinear = image.resample('bilinear');
```

If not, what does it actually do?